<!-- # CMPUT 200 Fall 2024  Ethics of Data Science and AI
 -->
# Assignment 2 - Employee Attrition Dataset : Applying and Analyzing Privacy Techniques

***
- **FIRST name**: Moffat
- **LAST name**: Muriithi
- **Student ID**:
- **Dataset**: employee_attrition.csv


In this assignment you will apply basic privacy techniques to a dataset of your choosing. By the end of this assignment, you should be able to:
1. Understand and implement randomized response on binary data;
2. Calculate the sensititivity of a non-binary feature;
3. Add noise to a non-binary feature;
4. Compute aggregate statistics.

---
# Assignment Guidelines

## Dataset selection

- You will choose **one dataset from a selection of four** and must
      use **the same dataset for both assignments**.

- Ensure that the dataset chosen matches the assignment version. See title for assignment version.

## Submission Details:

- Assignments must be submitted on **Canvas** by the due date at
      **11:59 PM**.

- Follow all instructions provided in the assignment notebooks.

- **File Naming:** You must name your submitted file as specified in
      the instructions. **Failure to do so will result in a 50%
      deduction** from your assignment marks, as stated in the Labs
      section.

- Ensure that all written code and answers are entered in the
      designated spaces in the notebook. If the autograder does not
      detect your responses, **you will not receive marks**, and this
      **cannot be negotiated after grading**.

## Late Submission Policy:**

- **10% penalty** for submissions after the deadline but within the
      first **24 hours** after the deadline.

- **20% total penalty** for submissions between **24 to 48 hours**
      after the deadline.

- **No submissions will be accepted beyond 48 hours** after the
      deadline.

- The late penalty is calculated as a percentage of the total
      assignment marks. **Example:** If an assignment is worth **120
      marks**, and you incur a **20% late penalty**, **24 marks** (20%
      of 120) will be deducted from your final score.

**Please ensure you adhere to these guidelines to avoid penalties.**

## Dataset descriptions

There are 4 datasets to choose from. Make sure to go through the
details of each one as a group before picking one.

1.  [Adult
    Dataset](https://archive.ics.uci.edu/dataset/2/adult):
    Used to predict an individual's income class based on census data.



3.  [Employee
    Attrition](https://www.kaggle.com/datasets/stealthtechnologies/employee-attrition-dataset?select=test.csv):
    Used to predict whether or not an employee will stay at their job
    based on demographic features and personal circumstances.

4.  [German
    Credit](https://online.stat.psu.edu/stat857/node/222/):
    Used to predict if an individual will be approved for a loan in
    Germany.


6.  [Law
    School](https://www.kaggle.com/datasets/danofer/law-school-admissions-bar-passage/data):
    Used to predict if a law student will pass the bar exam or not.



In [1]:
# Run this cell to set up; Please don't change this cell.

import numpy as np
from numpy.random import default_rng
rng = default_rng()
import pandas as pd
from scipy.optimize import minimize

# These lines do some fancy plotting magic.
import matplotlib
# This is a magic function that renders the figure in the notebook, instead of displaying a dump of the figure object.
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

## Part 1: Data

**Question 1.1.** We will begin by loading the Employee Attrition dataset into a DataFrame named df. This is the first step in our data inspection, preprocessing, and exploration.

In [2]:
# YOUR CODE HERE
df = pd.read_csv('employee_attrition.csv')

print("Shape: ", df.shape)
df.head(5)

Shape:  (14900, 24)


,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,...,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,52685,36,Male,13,Healthcare,8029,Excellent,High,Average,1,...,1,Mid,Large,22,No,No,No,Poor,Medium,Stayed
1,30585,35,Male,7,Education,4563,Good,High,Average,1,...,4,Entry,Medium,27,No,No,No,Good,High,Left
2,54656,50,Male,7,Education,5583,Fair,High,Average,3,...,2,Senior,Medium,76,No,No,Yes,Good,Low,Stayed
3,33442,58,Male,44,Media,5525,Fair,Very High,High,0,...,4,Entry,Medium,96,No,No,No,Poor,Low,Left
4,15667,39,Male,24,Education,4604,Good,High,Average,0,...,6,Mid,Large,45,Yes,No,No,Good,High,Stayed


In [3]:
# CELL USED FOR AUTOGRADER: do not delete!

**Question 1.2.** Describe your data and its purpose. Identify one variable that is binary or that could be classified into a binary feature. Identify another that is not a binary feature.

The Employee Attrition dataset contains demographic and professional details about employees. Its purpose is to predict whether an employee will leave the company based on these features. One variable that is a binary feature is Attrition (Stayed or Left). Age is not a binary feature.

## Part 2: Data pre-processing

**Question 2.1.** Clean the dataset by removing any rows with missing values. Then, convert the binary feature you identified in Question 1.2 into a numerical format (e.g., 0 and 1).

In [4]:
# YOUR CODE HERE
df = df.dropna()
df['Attrition'] = df['Attrition'].map({'Stayed': 1, 'Left': 0})
df

,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,...,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,52685,36,Male,13,Healthcare,8029,Excellent,High,Average,1,...,1,Mid,Large,22,No,No,No,Poor,Medium,1
1,30585,35,Male,7,Education,4563,Good,High,Average,1,...,4,Entry,Medium,27,No,No,No,Good,High,0
2,54656,50,Male,7,Education,5583,Fair,High,Average,3,...,2,Senior,Medium,76,No,No,Yes,Good,Low,1
3,33442,58,Male,44,Media,5525,Fair,Very High,High,0,...,4,Entry,Medium,96,No,No,No,Poor,Low,0
4,15667,39,Male,24,Education,4604,Good,High,Average,0,...,6,Mid,Large,45,Yes,No,No,Good,High,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14895,16243,56,Female,42,Healthcare,7830,Poor,Medium,Average,0,...,0,Senior,Medium,60,No,No,No,Poor,Medium,1
14896,47175,30,Female,15,Education,3856,Good,Medium,Average,2,...,0,Entry,Medium,20,No,No,No,Good,Medium,0
14897,12409,52,Male,5,Education,5654,Good,Very High,Below Average,0,...,4,Mid,Small,7,No,No,No,Good,High,0
14898,9554,18,Male,4,Education,5276,Fair,High,Average,0,...,3,Mid,Large,5,No,No,No,Poor,High,1


## Part 3: Randomized Response

Now let's implement a Randomized Response mechanism. First we have to identify what the query on our **binary** variable will be. Then we can create our own randomized mechanism.

**Question 3.1.** Write a query on your binary feature.

Are you planning on staying at the company?

**Question 3.2.** Create your own randomized response mechanism for the query you defined above. You may NOT use the coin example from class, try to be creative!

By using cards, can place it that when a card is drawn:
If the card is a club(0.25), always answer Yes (1).
If the card is a spade(0.25), always answer No (0).
If the card is a heart or diamond(0.50), the drawer of the card should answer truthfully.

**Question 3.3.** Implement a function for your mechanism in 3.2. in the code cell below. The function should accept a value `0` or `1` and return as output the reported answer according to the randomized response above. Name your function `rand_resp`.

In [5]:
# YOUR CODE HERE
def rand_resp(true_val):
    draw = rng.random()
    if draw < 0.50:
        return true_val
    elif draw < 0.75:
        return 1
    else:
        return 0

**Question 3.4.** For each value in your dataframe's binary feature column, call your function. Store the results in a new column in df named `rrc1`.

In [6]:
# YOUR CODE HERE
df['rrc1'] = df['Attrition'].apply(rand_resp)

**Question 3.5.** Now get the **estimate** for the true number of people who answered `1`.  Write this result into the variable `count_est_true_yes`.  Given the number of reported `1`'s, we know how to estimate the proportion or number of true `1`'s.

Calculate the true number of people who answered `1` (the true answer in the data we imported) and write it into the variable `count_true_yes`.

In [7]:
# YOUR CODE HERE
total_responses = len(df)
reported_yes_count = df['rrc1'].sum()

p_star = reported_yes_count / total_responses
p_true_est = (p_star - 0.25) / 0.50

count_est_true_yes = int(p_true_est * total_responses)
count_true_yes = df['Attrition'].sum()
print("Estimated true yes count: ", count_est_true_yes)
print("True yes count: ", count_true_yes)

Estimated true yes count:  7840
True yes count:  7868


**Question 3.6.** Comment on your results from above. What can you say about the privacy-accuracy tradeoff of your randomized response mechanism?

The response mechanism provides privacy because any "Yes" response could easily just be the result of drawing a Club. However, this injected noise reduces the accuracy.

**Question 3.7.** We learned in class that data analysts are still able to obtain aggregate statistics from the results of a randomized response survey. Using the column you made above, `rrc1`, calculate the mean and median for the estimated true responses. Do the same for your true responses. Name your answers `mean_est_true_yes`, `mean_true_yes`, `median_est_true_yes`, and `median_true_yes` respectively.

In [8]:
# YOUR CODE HERE
mean_true_yes = df['Attrition'].mean()
median_true_yes = df['Attrition'].median()

# For the estimated true responses, the mean is our estimated true probability
mean_est_true_yes = (df['rrc1'].mean() - 0.25) / 0.50

median_est_true_yes = 1 if mean_est_true_yes > 0.5 else 0
print("Mean: ", mean_est_true_yes, mean_true_yes)
print("Median: ", median_est_true_yes, median_true_yes)

Mean:  0.5261744966442954 0.5280536912751678
Median:  1 1.0


**Question 3.8.** Comment on your results from above. Are the results from your mechanism useable? What can you say about the privacy when it comes to randomized response? Comment on the distributions of your data.

The mechanism's results are usable for aggregate data analysis asestimated mean tracks closely with the true mean. As individual records are highly private, the overall distribution of the data remains intact, allowing data scientists to still derive insights about the population without risking confidentiality.

## Part 4: Adding Noise

We are going to use the non-binary feature that you chose earlier and add noise to it. To do this, we apply the same steps that we would for differential privacy: $f(D) + Z$.

**Question 4.1.** Suppose the function we wish to query is **count**. What would the global sensitivity, $S(f)$, be? Explain why.

The global sensitivity, S(f), of a count query is 1.
Global sensitivity is the maximum amount the query output could change if a single record in the dataset is added or removed. Adding or removing one employee from the database can only change the total count of any given category by exactly 1.

**Question 4.2.** In the cell below, write a function that adds Laplace noise to a value given the sensitivity and privacy parameter epsilon. Name your function `add_laplace_noise`.

In [9]:
# YOUR CODE HERE
def add_laplace_noise(value, sensitivity, epsilon):
    scale = sensitivity / epsilon
    noise = rng.laplace(loc=0.0, scale=scale) 
    return value + noise

Since our query is on count, we'll have to obtain the count of each unique value of our feature.

**Question 4.3.** Define a variable called `feature_counts` that holds the count of each unique value of your feature. *Hint: there's a method that does this for us!*

In [11]:
# YOUR CODE HERE
feature_counts = df['Job Role'].value_counts()

**Question 4.4.** Before we add noise, let's calculate some stats for the `feature_counts` variable you defined in Question 4.3. Calculate the mean, median, and count and name them `mean_count`, `median_count`, and `total_count` respectively.

In [12]:
# YOUR CODE HERE
mean_count = feature_counts.mean()
median_count = feature_counts.median()
total_count = feature_counts.sum()

print(f"Mean of true counts: {mean_count}")
print(f"Median of true counts: {median_count}")
print(f"Total count of true records: {total_count}")

Mean of true counts: 2980.0
Median of true counts: 3168.0
Total count of true records: 14900


**Question 4.5.** Now we can start adding noise. Use your Laplace function and your variable from 4.3.to calculate a noisy representation of each value in your feature. Set your value of epsilon to 1 for now.

In [13]:
# YOUR CODE HERE
epsilon = 1
sensitivity = 1

noisy_counts = feature_counts.apply(lambda x: add_laplace_noise(x, sensitivity, epsilon))

**Question 4.6.** Now calculate the stats for your noisy values. Calculate the mean, median, and count and name them `mean_noisy_count`, `median_noisy_count`, and `total_noisy_count` respectively.

In [14]:
# YOUR CODE HERE
mean_noisy_count = noisy_counts.mean()
median_noisy_count = noisy_counts.median()
total_noisy_count = noisy_counts.sum()

print(f"Noisy mean of true counts: {mean_noisy_count}")
print(f"Noisy median of true counts: {median_noisy_count}")
print(f"Total noisy count of true records: {total_noisy_count}")

Noisy mean of true counts: 2979.6940958587725
Noisy median of true counts: 3167.7550352974376
Total noisy count of true records: 14898.470479293863


**Question 4.7.** Comment on the differences in aggregate statistics between the original and the noisy values. What can you say about the utility and privacy?  

The aggregate statistics of the noisy data remain quite close to the original data, meaning utility is largely preserved. Because Laplace noise is zero-centered, the noise often cancels itself out when aggregating across the entire dataset (e.g., calculating the total). This protects individual departmental counts without ruining the big picture.

**Question 4.8.** Go back to question 4.5. and change the value of epsilon. Repeat this until you notice a pattern in your aggregate statistics. What happens as epsilon changes? What happens to the privacy and the accuracy?

Changing epsilon to 0.1 makes the noise becomes much larger. Changing it to 10, the noise becomes tiny.
As epsilon decreases, the scale of the Laplace distribution increases, meaning more noise is added. This results in higher privacy but lower accuracy. Conversely, as epsilon increases, less noise is added, leading to lower privacy but higher accuracy.

# Rubric

| Question | Points|
|----------|----------|
| 1.1.   | 5   |
| 1.2.    | 10   |
| 2.1.    | 5   |
| 3.1.   | 5   |
| 3.2.    | 10  |
| 3.3.  | 10   |
| 3.4.    | 2   |
| 3.5.   | 8   |
| 3.6.   | 5   |
| 3.7.   | 6   |
| 3.8.   | 8   |
| 4.1.   | 2   |
| 4.2.    | 10  |
| 4.3.  | 5   |
| 4.4.    | 3   |
| 4.5.   | 5   |
| 4.6.   | 3   |
| 4.7.   | 5   |
| 4.8.   | 8   |
| Total  | 115   |


